# AFC Wimbledon — Form Analysis

Exploring match results, form streaks, and goal difference over time.

Requires `data/results.json` — run `python src/fetch_results.py` first.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/results.json")

with open(DATA_PATH) as f:
    results = json.load(f)

df = pd.DataFrame(results)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df.head()

## Cumulative Points Over the Season

In [ ]:
df["points"] = df["result"].map({"W": 3, "D": 1, "L": 0})
df["cumulative_pts"] = df["points"].cumsum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["date"], df["cumulative_pts"], marker="o", markersize=4)
ax.set_title("Cumulative Points")
ax.set_ylabel("Points")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Goal Difference Over Time

In [ ]:
# Per-match GD from AFC Wimbledon's perspective
df["gf"] = df.apply(lambda r: r["home_score"] if r["home"] == "AFC Wimbledon" else r["away_score"], axis=1)
df["ga"] = df.apply(lambda r: r["away_score"] if r["home"] == "AFC Wimbledon" else r["home_score"], axis=1)
df["gd"] = df["gf"] - df["ga"]
df["cumulative_gd"] = df["gd"].cumsum()

fig, ax = plt.subplots(figsize=(10, 4))
colors = df["gd"].apply(lambda x: "green" if x > 0 else ("red" if x < 0 else "gray"))
ax.bar(df["date"], df["gd"], color=colors, width=2)
ax.plot(df["date"], df["cumulative_gd"], color="black", linewidth=1.5, label="Cumulative GD")
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_title("Goal Difference per Match")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Home vs Away Record

In [ ]:
df["venue"] = df["home"].apply(lambda h: "Home" if h == "AFC Wimbledon" else "Away")

home_away = df.groupby(["venue", "result"]).size().unstack(fill_value=0)
home_away = home_away.reindex(columns=["W", "D", "L"], fill_value=0)

fig, ax = plt.subplots(figsize=(6, 4))
home_away.plot(kind="bar", ax=ax, color=["green", "gold", "red"])
ax.set_title("Home vs Away Results")
ax.set_ylabel("Matches")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Result")
plt.tight_layout()
plt.show()

## Rolling Form (5-match window)

In [ ]:
df["rolling_pts"] = df["points"].rolling(5, min_periods=1).mean() * 3  # points per game scaled to 3

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(df["date"], df["rolling_pts"], alpha=0.3, color="blue")
ax.plot(df["date"], df["rolling_pts"], color="blue", linewidth=1.5)
ax.axhline(2.0, color="green", linestyle="--", alpha=0.5, label="Promotion pace (~2 PPG)")
ax.axhline(1.0, color="red", linestyle="--", alpha=0.5, label="Relegation pace (~1 PPG)")
ax.set_title("Rolling Points per Game (5-match window)")
ax.set_ylabel("PPG (scaled to 3)")
ax.set_ylim(0, 3)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()